# 01 EDA Main Table

Initial exploration of `application_train.csv` and `application_test.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file_path = "application_train.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes.value_counts())

print("\nFirst 5 rows:")
print(df.head())

print("\nTarget distribution:")
print(df["TARGET"].value_counts())
print(df["TARGET"].value_counts(normalize=True))

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumber of numeric columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

# Missing values
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percent": missing_percent
}).sort_values(by="Missing Percent", ascending=False)

print("\nTop 20 columns with missing values:")
print(missing_df[missing_df["Missing Count"] > 0].head(20))

plt.figure(figsize=(10, 6))
missing_df[missing_df["Missing Count"] > 0].head(20)["Missing Percent"].sort_values().plot(kind="barh")
plt.title("Top 20 Columns by Missing Percentage")
plt.xlabel("Missing Percentage")
plt.tight_layout()
plt.show()

# Target class balance
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="TARGET")
plt.title("Class Distribution of TARGET")
plt.xlabel("TARGET")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


# Basic statistics for numeric columns
print("\nSummary statistics for numeric columns:")
print(df[numeric_cols].describe())


# Histograms for important numeric features
features_to_plot = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE"
]

for col in features_to_plot:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col].dropna(), bins=50)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


# Create age and employment years for readability
df["YEARS_BIRTH"] = (-df["DAYS_BIRTH"]) / 365
df["YEARS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["YEARS_EMPLOYED"] = (-df["YEARS_EMPLOYED"]) / 365

plt.figure(figsize=(7, 4))
sns.histplot(df["YEARS_BIRTH"].dropna(), bins=40)
plt.title("Distribution of Age in Years")
plt.xlabel("Age")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.histplot(df["YEARS_EMPLOYED"].dropna(), bins=40)
plt.title("Distribution of Employment Length in Years")
plt.xlabel("Years Employed")
plt.tight_layout()
plt.show()

# Numeric features vs target
boxplot_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "YEARS_BIRTH"
]

for col in boxplot_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="TARGET", y=col)
    plt.title(f"{col} by TARGET")
    plt.tight_layout()
    plt.show()


# Default rate by important categorical features
cat_features_to_plot = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE"
]

for col in cat_features_to_plot:
    default_rate = df.groupby(col)["TARGET"].mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    default_rate.plot(kind="bar")
    plt.title(f"Default Rate by {col}")
    plt.ylabel("Mean TARGET")
    plt.tight_layout()
    plt.show()

# Correlation heatmap for selected numeric columns
selected_corr_cols = [
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "REGION_POPULATION_RELATIVE",
    "YEARS_BIRTH",
    "YEARS_EMPLOYED",
    "CNT_CHILDREN"
]

corr = df[selected_corr_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()